<a href="https://colab.research.google.com/github/potato-yen/114-2-Programming-Language/blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 日常支出速算與分攤（作業一）

**目標**：從 Google Sheet 讀取消費紀錄 → 計算總額 / 分類小計 / AA 分攤。

Sheet 欄位：`日期` `時間` `分類` `品項` `金額` `付款人` `地點` `支付方式` `備註`

GoogleSheet: https://docs.google.com/spreadsheets/d/1Etj3aomNgs1hroV3JwgRMNMVQ2_BIyKwChgtPRnYxic/edit?usp=sharing

In [1]:
!pip install -q gradio gspread

In [2]:
import gradio as gr
import pandas as pd
import datetime
import gspread
from google.colab import auth
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print('✓ Google 驗證完成')

✓ Google 驗證完成


In [3]:
# ── 設定 ──────────────────────────────────────────────────────────
SHEET_URL      = "https://docs.google.com/spreadsheets/d/1Etj3aomNgs1hroV3JwgRMNMVQ2_BIyKwChgtPRnYxic/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"
REQUIRED_COLS  = ["日期", "時間", "分類", "品項", "金額", "付款人", "地點", "支付方式", "備註"]

_ws = None

def _get_ws():
    global _ws
    if _ws is None:
        gs  = gc.open_by_url(SHEET_URL)
        _ws = gs.worksheet(WORKSHEET_NAME)
        rows = _ws.get_all_values()
        if not rows:
            _ws.append_row(REQUIRED_COLS, value_input_option="USER_ENTERED")
        elif rows[0] != REQUIRED_COLS:
            _ws.update('A1', [REQUIRED_COLS])
    return _ws

def _read_df():
    ws   = _get_ws()
    vals = ws.get_all_values()
    if len(vals) <= 1:
        return pd.DataFrame(columns=REQUIRED_COLS)
    df = pd.DataFrame(vals[1:], columns=vals[0])
    df = df.reindex(columns=REQUIRED_COLS, fill_value="")
    df["金額"] = pd.to_numeric(df["金額"], errors="coerce").fillna(0.0)
    return df

def _summary(df):
    total = float(df["金額"].sum()) if not df.empty else 0.0
    count = len(df)
    avg   = total / count if count > 0 else 0.0

    by_cat = (
        df.groupby("分類", as_index=False)["金額"]
          .sum()
          .sort_values("金額", ascending=False)
          .rename(columns={"金額": "金額 (NT$)"})
        if not df.empty
        else pd.DataFrame(columns=["分類", "金額 (NT$)"])
    )

    if not df.empty and df["付款人"].replace("", pd.NA).notna().any():
        payers = df["付款人"].replace("", "匿名").fillna("匿名")
        uniq   = sorted(p for p in payers.unique() if p)
        share  = round(total / max(len(uniq), 1), 1)
        paid   = df.groupby("付款人", as_index=False)["金額"].sum().rename(columns={"金額": "實付"})
        settle = (
            pd.DataFrame({"付款人": uniq})
              .merge(paid, on="付款人", how="left")
              .fillna({"實付": 0.0})
        )
        settle["應付 (AA)"] = share
        settle["差額"]      = (settle["實付"] - settle["應付 (AA)"]).round(1)
        settle = settle.sort_values("差額", ascending=False)
    else:
        settle = pd.DataFrame(columns=["付款人", "實付", "應付 (AA)", "差額"])

    return total, count, avg, by_cat, settle

# ── HTML 渲染工具 ─────────────────────────────────────────────────
def _df_to_html(df, money_cols=None, diff_col=None, empty_msg="尚無資料"):
    """將 DataFrame 轉為純 HTML 表格，不含 Gradio 的排序選單。"""
    if df.empty:
        return f'<div class="table-empty">{empty_msg}</div>'
    money_cols = money_cols or []
    rows_html  = ""
    for _, row in df.iterrows():
        cells = ""
        for col in df.columns:
            val = row[col]
            extra_cls = ""
            if col in money_cols and isinstance(val, (int, float)):
                display = f"NT$ {val:,.0f}"
                extra_cls = " td-num"
            elif col == diff_col and isinstance(val, (int, float)):
                display = f"{val:+,.0f}"
                extra_cls = " td-pos" if val >= 0 else " td-neg"
            else:
                display = str(val) if val != "" else "—"
            cells += f'<td class="td{extra_cls}">{display}</td>'
        rows_html += f"<tr>{cells}</tr>"
    headers = "".join(f'<th class="th">{c}</th>' for c in df.columns)
    return f"""
<div class="tbl-wrap">
  <table class="tbl">
    <thead><tr>{headers}</tr></thead>
    <tbody>{rows_html}</tbody>
  </table>
</div>"""

def _kpi_html(total, count, avg):
    return f"""
<div class="kpi-row">
  <div class="kpi-card kpi-main">
    <span class="kpi-label">總支出</span>
    <span class="kpi-val kpi-accent">NT$ {total:,.0f}</span>
  </div>
  <div class="kpi-card">
    <span class="kpi-label">記帳筆數</span>
    <span class="kpi-val">{count}</span>
  </div>
  <div class="kpi-card">
    <span class="kpi-label">平均每筆</span>
    <span class="kpi-val">NT$ {avg:,.0f}</span>
  </div>
</div>"""

def _toast(msg, ok=True):
    cls  = "toast-ok" if ok else "toast-err"
    icon = "✓" if ok else "✗"
    return f'<div class="toast {cls}">{icon}&ensp;{msg}</div>'

# ── Handlers ─────────────────────────────────────────────────────
def add_expense(date_s, time_s, category, item, amount, payer, location, pay_method, note):
    try:
        if not date_s:
            date_s = datetime.date.today().strftime("%Y-%m-%d")
        try:
            amt = float(str(amount).replace(",", ""))
        except:
            return _toast("金額格式錯誤，請輸入數字", ok=False), gr.update(), gr.update(), gr.update(), gr.update()

        ws = _get_ws()
        ws.append_row(
            [date_s, time_s or "", (category or "其他").strip(),
             (item or "未填").strip(), amt, (payer or "匿名").strip(),
             location or "", pay_method or "", note or ""],
            value_input_option="USER_ENTERED"
        )
        df = _read_df()
        total, count, avg, by_cat, settle = _summary(df)
        recent = df.iloc[::-1].head(8).reset_index(drop=True)
        return (
            _toast(f"已記錄 ── {category} · {item} · NT${amt:,.0f}"),
            _kpi_html(total, count, avg),
            _df_to_html(by_cat, money_cols=["金額 (NT$)"]),
            _df_to_html(settle, money_cols=["實付", "應付 (AA)"], diff_col="差額"),
            _df_to_html(recent, money_cols=["金額"])
        )
    except Exception as e:
        return _toast(str(e), ok=False), gr.update(), gr.update(), gr.update(), gr.update()

def refresh():
    try:
        df = _read_df()
        total, count, avg, by_cat, settle = _summary(df)
        recent = df.iloc[::-1].head(8).reset_index(drop=True)
        return (
            _kpi_html(total, count, avg),
            _df_to_html(by_cat, money_cols=["金額 (NT$)"]),
            _df_to_html(settle, money_cols=["實付", "應付 (AA)"], diff_col="差額"),
            _df_to_html(recent, money_cols=["金額"])
        )
    except Exception as e:
        err = _toast(str(e), ok=False)
        return err, err, err, err

print("✓ 函式定義完成")

✓ 函式定義完成


In [4]:
# ── CSS ───────────────────────────────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500&family=DM+Mono:wght@300;400&family=Noto+Sans+TC:wght@300;400;500&display=swap');

:root {
  --bg:          #F7F5F2;
  --surface:     #FFFFFF;
  --surface2:    #F0EDE8;
  --border:      #E5E0D8;
  --border-soft: rgba(0,0,0,0.06);
  --accent:      #D97706;
  --accent-dim:  rgba(217,119,6,0.08);
  --text:        #1C1917;
  --text-2:      #57534E;
  --muted:       #A8A29E;
  --green:       #16A34A;
  --green-bg:    rgba(22,163,74,0.07);
  --red:         #DC2626;
  --red-bg:      rgba(220,38,38,0.07);
  --radius:      8px;
  --radius-sm:   5px;
  --mono:        'DM Mono', monospace;
  --sans:        'DM Sans', 'Noto Sans TC', sans-serif;
  --shadow:      0 1px 3px rgba(0,0,0,0.06), 0 1px 2px rgba(0,0,0,0.04);
  --shadow-md:   0 4px 12px rgba(0,0,0,0.08);
}

/* ── Root ── */
body, .gradio-container {
  background: var(--bg) !important;
  font-family: var(--sans) !important;
  color: var(--text) !important;
}
.gradio-container { max-width: 1280px !important; margin: 0 auto !important; padding: 0 !important; }
.gradio-container .block.padded { background: transparent !important; border: none !important; padding: 0 !important; }
.gradio-container .form         { border: none !important; background: transparent !important; }

/* ── App Header ── */
.app-header {
  padding: 32px 40px 24px;
  border-bottom: 1px solid var(--border);
  display: flex;
  align-items: baseline;
  gap: 14px;
  background: var(--surface);
}
.app-header h1 {
  font-family: var(--sans);
  font-size: 22px;
  font-weight: 500;
  color: var(--text);
  letter-spacing: -0.01em;
  margin: 0;
}
.app-header .dot { color: var(--accent); font-size: 20px; line-height: 1; }
.app-header .sub {
  font-family: var(--mono);
  font-size: 10.5px;
  color: var(--muted);
  letter-spacing: 0.1em;
  text-transform: uppercase;
}

/* ── Main layout ── */
#main-wrap { padding: 28px 40px 48px; gap: 24px !important; }

/* ── Left panel (form) ── */
.form-panel {
  background: var(--surface);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  padding: 28px;
  box-shadow: var(--shadow);
}
.panel-label {
  font-size: 10px;
  font-family: var(--mono);
  letter-spacing: 0.13em;
  text-transform: uppercase;
  color: var(--muted);
  margin-bottom: 18px;
  padding-bottom: 14px;
  border-bottom: 1px solid var(--border);
}

/* ── Quick-pick pills ── */
.pills-outer { margin-bottom: 18px; }
.pills-label {
  font-size: 10px;
  font-family: var(--mono);
  letter-spacing: 0.1em;
  text-transform: uppercase;
  color: var(--muted);
  margin-bottom: 8px;
}
.pills-row { display: flex; flex-wrap: wrap; gap: 6px; }
.pill {
  padding: 4px 11px;
  border-radius: 20px;
  border: 1px solid var(--border);
  font-family: var(--sans);
  font-size: 12px;
  color: var(--text-2);
  cursor: pointer;
  background: var(--surface);
  transition: all 0.13s ease;
  user-select: none;
}
.pill:hover {
  border-color: var(--accent);
  color: var(--accent);
  background: var(--accent-dim);
}

/* ── Section dividers ── */
.section-label {
  font-size: 10px;
  font-family: var(--mono);
  letter-spacing: 0.13em;
  text-transform: uppercase;
  color: var(--muted);
  margin: 26px 0 12px;
  padding-top: 22px;
  border-top: 1px solid var(--border);
}
.section-label:first-child { margin-top: 0; padding-top: 0; border-top: none; }

/* ── KPI row ── */
.kpi-row {
  display: flex;
  gap: 12px;
  margin-bottom: 24px;
}
.kpi-card {
  flex: 1;
  background: var(--surface);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  padding: 18px 20px;
  display: flex;
  flex-direction: column;
  gap: 5px;
  box-shadow: var(--shadow);
}
.kpi-main { flex: 1.5; }
.kpi-label {
  font-size: 10px;
  font-family: var(--mono);
  letter-spacing: 0.12em;
  text-transform: uppercase;
  color: var(--muted);
}
.kpi-val {
  font-family: var(--mono);
  font-size: 26px;
  font-weight: 300;
  color: var(--text);
  letter-spacing: -0.02em;
}
.kpi-accent { color: var(--accent); }

/* ── Custom HTML tables ── */
.tbl-wrap {
  overflow-x: auto;
  border: 1px solid var(--border);
  border-radius: var(--radius);
  background: var(--surface);
}
.tbl {
  width: 100%;
  border-collapse: collapse;
  font-family: var(--mono);
  font-size: 12.5px;
}
.th {
  background: var(--surface2);
  color: var(--muted);
  font-size: 10px;
  letter-spacing: 0.1em;
  text-transform: uppercase;
  padding: 10px 14px;
  text-align: left;
  font-weight: 400;
  border-bottom: 1px solid var(--border);
  white-space: nowrap;
}
.td {
  padding: 9px 14px;
  color: var(--text);
  border-bottom: 1px solid var(--border-soft);
  font-size: 12.5px;
  white-space: nowrap;
}
.tbl tbody tr:last-child .td { border-bottom: none; }
.tbl tbody tr:hover td { background: var(--surface2); }
.td-num { text-align: right; color: var(--text-2); }
.td-pos { text-align: right; color: var(--green); font-weight: 500; }
.td-neg { text-align: right; color: var(--red); font-weight: 500; }
.table-empty {
  padding: 24px;
  text-align: center;
  color: var(--muted);
  font-family: var(--mono);
  font-size: 12px;
  border: 1px dashed var(--border);
  border-radius: var(--radius);
  letter-spacing: 0.06em;
}

/* ── Toast ── */
.toast {
  padding: 11px 15px;
  border-radius: var(--radius-sm);
  font-family: var(--mono);
  font-size: 12px;
  letter-spacing: 0.03em;
  animation: fadeSlide 0.22s ease;
  margin-bottom: 4px;
}
.toast-ok  { background: var(--green-bg); border: 1px solid rgba(22,163,74,0.2); color: var(--green); }
.toast-err { background: var(--red-bg);   border: 1px solid rgba(220,38,38,0.2); color: var(--red); }
@keyframes fadeSlide {
  from { opacity: 0; transform: translateY(-5px); }
  to   { opacity: 1; transform: none; }
}

/* ── Gradio input overrides ── */
.gradio-container input[type=text],
.gradio-container input[type=number],
.gradio-container textarea {
  background: var(--surface) !important;
  border: 1px solid var(--border) !important;
  border-radius: var(--radius-sm) !important;
  color: var(--text) !important;
  font-family: var(--mono) !important;
  font-size: 13px !important;
  transition: border-color 0.15s, box-shadow 0.15s;
  box-shadow: none !important;
}
.gradio-container input:focus,
.gradio-container textarea:focus {
  border-color: var(--accent) !important;
  outline: none !important;
  box-shadow: 0 0 0 3px var(--accent-dim) !important;
}
.gradio-container label > span,
.gradio-container .label-wrap span {
  font-family: var(--mono) !important;
  font-size: 10px !important;
  letter-spacing: 0.1em !important;
  text-transform: uppercase !important;
  color: var(--muted) !important;
  font-weight: 400 !important;
}

/* ── Submit button ── */
#btn-add {
  background: var(--accent) !important;
  color: #FFFFFF !important;
  border: none !important;
  border-radius: var(--radius-sm) !important;
  font-family: var(--sans) !important;
  font-size: 13px !important;
  font-weight: 500 !important;
  letter-spacing: 0.02em !important;
  padding: 12px 0 !important;
  width: 100% !important;
  cursor: pointer;
  transition: opacity 0.15s, transform 0.1s;
  box-shadow: 0 1px 2px rgba(0,0,0,0.1) !important;
  margin-top: 6px;
}
#btn-add:hover  { opacity: 0.9; transform: translateY(-1px); box-shadow: var(--shadow-md) !important; }
#btn-add:active { transform: translateY(0); opacity: 1; }

/* ── Refresh button ── */
#btn-refresh {
  background: transparent !important;
  border: 1px solid var(--border) !important;
  color: var(--muted) !important;
  border-radius: var(--radius-sm) !important;
  font-family: var(--mono) !important;
  font-size: 10px !important;
  letter-spacing: 0.1em !important;
  text-transform: uppercase !important;
  padding: 9px 18px !important;
  cursor: pointer;
  transition: border-color 0.13s, color 0.13s;
  margin-top: 16px;
}
#btn-refresh:hover { border-color: var(--accent) !important; color: var(--accent) !important; }

/* ── Accordion ── */
.gradio-container details summary {
  font-family: var(--mono) !important;
  font-size: 11px !important;
  letter-spacing: 0.08em !important;
  color: var(--muted) !important;
  cursor: pointer;
}
.gradio-container details {
  border: 1px solid var(--border) !important;
  border-radius: var(--radius-sm) !important;
  padding: 10px 14px !important;
  background: var(--surface2) !important;
  margin-top: 10px;
}

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 4px; height: 4px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: var(--border); border-radius: 99px; }
"""

# ── Gradio UI ─────────────────────────────────────────────────────
with gr.Blocks(
    title="支出帳本",
    css=CSS,
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.orange,
        neutral_hue=gr.themes.colors.stone,
    )
) as demo:

    gr.HTML("""
    <div class="app-header">
      <span class="dot">●</span>
      <h1>支出帳本</h1>
      <span class="sub">Daily Expense Tracker &amp; AA Split</span>
    </div>
    """)

    with gr.Row(elem_id="main-wrap"):

        # ── Left: Form
        with gr.Column(scale=5):
            gr.HTML('<div class="form-panel"><!-- wrap start --></div>')
            gr.HTML('<div class="panel-label">新增記錄</div>')

            gr.HTML("""
            <div class="pills-outer">
              <div class="pills-label">快選分類</div>
              <div class="pills-row">
                <span class="pill" onclick="var el=document.querySelector('#cat-input input');el.value=this.textContent;el.dispatchEvent(new Event('input'));">外食</span>
                <span class="pill" onclick="var el=document.querySelector('#cat-input input');el.value=this.textContent;el.dispatchEvent(new Event('input'));">飲料</span>
                <span class="pill" onclick="var el=document.querySelector('#cat-input input');el.value=this.textContent;el.dispatchEvent(new Event('input'));">交通</span>
                <span class="pill" onclick="var el=document.querySelector('#cat-input input');el.value=this.textContent;el.dispatchEvent(new Event('input'));">購物</span>
                <span class="pill" onclick="var el=document.querySelector('#cat-input input');el.value=this.textContent;el.dispatchEvent(new Event('input'));">娛樂</span>
                <span class="pill" onclick="var el=document.querySelector('#cat-input input');el.value=this.textContent;el.dispatchEvent(new Event('input'));">生活雜費</span>
                <span class="pill" onclick="var el=document.querySelector('#cat-input input');el.value=this.textContent;el.dispatchEvent(new Event('input'));">醫療</span>
              </div>
            </div>
            """)

            with gr.Row():
                date_in = gr.Textbox(label="日期", value=datetime.date.today().strftime("%Y-%m-%d"), scale=3)
                time_in = gr.Textbox(label="時間", value=datetime.datetime.now().strftime("%H:%M"), scale=2)

            cat_in  = gr.Textbox(label="分類",  placeholder="外食 / 交通 / 購物…", elem_id="cat-input")
            item_in = gr.Textbox(label="品項",  placeholder="咖啡 / 便當 / 車票…")

            with gr.Row():
                amt_in   = gr.Textbox(label="金額 (NT$)", placeholder="0", scale=2)
                payer_in = gr.Textbox(label="付款人",      placeholder="Amethus / Alice…", scale=3)

            with gr.Accordion("更多欄位（可選）", open=False):
                loc_in    = gr.Textbox(label="地點",   placeholder="7-11 / 全聯…")
                method_in = gr.Textbox(label="支付方式", placeholder="現金 / 信用卡 / Apple Pay…")
                note_in   = gr.Textbox(label="備註",   placeholder="補充說明")

            btn_add = gr.Button("記帳", elem_id="btn-add")
            toast   = gr.HTML()

        # ── Right: Dashboard
        with gr.Column(scale=7):
            kpi_html  = gr.HTML(_kpi_html(0, 0, 0))

            gr.HTML('<div class="section-label">分類小計</div>')
            cat_tbl   = gr.HTML()

            gr.HTML('<div class="section-label">AA 分攤結算</div>')
            settle_tbl = gr.HTML()

            gr.HTML('<div class="section-label">最近記錄（最新 8 筆）</div>')
            recent_tbl = gr.HTML()

            btn_refresh = gr.Button("↻  重新讀取", elem_id="btn-refresh")

    # ── Events
    btn_add.click(
        fn=add_expense,
        inputs=[date_in, time_in, cat_in, item_in, amt_in,
                payer_in, loc_in, method_in, note_in],
        outputs=[toast, kpi_html, cat_tbl, settle_tbl, recent_tbl]
    )
    btn_refresh.click(
        fn=refresh, inputs=[],
        outputs=[kpi_html, cat_tbl, settle_tbl, recent_tbl]
    )
    demo.load(
        fn=refresh, inputs=[],
        outputs=[kpi_html, cat_tbl, settle_tbl, recent_tbl]
    )

demo.launch(share=True)

/tmp/ipykernel_3128/3601201556.py:316: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_3128/3601201556.py:316: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7873cc430eced30b3f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
